# Track A Individual Baseline


Step 1. 업로드한 파일 확인


In [1]:

from pathlib import Path
import sys

# Local WSL/Jupyter 기준 설정입니다.
# 이 노트북은 MainQuest/Quest02 폴더에서 실행하는 것을 기본으로 합니다.
def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for base in candidates:
        if (base / "data" / "RS18A").exists():
            return base
        quest02 = base / "MainQuest" / "Quest02"
        if (quest02 / "data" / "RS18A").exists():
            return quest02
    return cwd

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "RS18A"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "trackA_individual"
BASELINE_TRAIN_CSV_PATH = OUTPUT_DIR / "baseline_train_labels.csv"
MODEL_PATH = OUTPUT_DIR / "model_individual.pt"
SUBMISSION_PATH = OUTPUT_DIR / "submission_a_individual.csv"
PYTHON_EXECUTABLE = sys.executable

# 인터넷 다운로드 없이 실행 가능하게 기본은 ImageNet pretrained weight를 사용하지 않습니다.
# 이미 로컬 캐시가 있고 pretrained를 쓰고 싶으면 True로 바꿔도 됩니다.
USE_PRETRAINED = False
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-4
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.2
SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [
    DATA_DIR,
    DATA_DIR / "train",
    DATA_DIR / "test",
    DATA_DIR / "train_labels.csv",
    DATA_DIR / "sample_submission.csv",
]

for path in required_paths:
    print(path, "존재:", path.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


/home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A 존재: True
/home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/train 존재: True
/home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/test 존재: True
/home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/train_labels.csv 존재: True
/home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/sample_submission.csv 존재: True
OUTPUT_DIR: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_individual


In [2]:

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("현재 Quest02 폴더 파일:")
for p in sorted(PROJECT_ROOT.iterdir()):
    print("-", p.name)


PROJECT_ROOT: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02
DATA_DIR: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A
OUTPUT_DIR: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_individual
현재 Quest02 폴더 파일:
- 0630.ipynb
- 18B.csv
- 50B.csv
- Archives
- README.md
- base0_resnet18_finetune.ipynb
- base0_resnet18_finetune_prep.ipynb
- base0_resnet50.ipynb
- checkpoints
- data
- examples.ipynb
- labeled
- logs
- model
- outlier_dup_pair_stats.ipynb
- outputs
- predict_rs18a_resnet18_50.ipynb
- rawresult.csv
- rawresult_resnet18_finetune.csv
- submissionA_resnet18_finetune.csv
- submissionA_resnet18_finetune_nocrop.csv
- submissionB_ood_demote_visible_falsepos.csv
- submissionB_score_distribution.ipynb
- trackA_individual.ipynb
- trackA_individual_prep.ipynb
- trackA_individual_prep.ipynb:Zone.Identifier
- trackA_retrain.ipynb
- trackA_retrain_070202.ipynb
- trackB_070101.ipynb
- trackB_070102.ipynb
- trackB_070103.ipynb
- trackB_070104.ipynb
- 

Step 2. 외부 보조 파일 없이 데이터 경로 확인하기


In [3]:

# 이 버전은 조원 보조 zip/script 없이 노트북 안에서 바로 학습/예측합니다.
# 필요한 입력은 data/RS18A 안의 train, test, train_labels.csv, sample_submission.csv뿐입니다.
print("helper script 사용 안 함")
print("train_labels:", DATA_DIR / "train_labels.csv")
print("sample_submission:", DATA_DIR / "sample_submission.csv")


helper script 사용 안 함
train_labels: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/train_labels.csv
sample_submission: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/sample_submission.csv


Step 3. RS18A 데이터가 모두 준비됐는지 다시 확인하기


In [4]:

required_paths = [
    DATA_DIR / "train",
    DATA_DIR / "test",
    DATA_DIR / "train_labels.csv",
    DATA_DIR / "sample_submission.csv",
]

missing = [path for path in required_paths if not path.exists()]

for path in required_paths:
    print(path.name, "존재:", path.exists(), "|", path)

if missing:
    print("누락된 데이터 파일이 있습니다. 아래 경로를 확인하세요:")
    for path in missing:
        print("-", path)
else:
    print("필요한 RS18A 데이터 파일이 모두 있습니다.")


train 존재: True | /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/train
test 존재: True | /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/test
train_labels.csv 존재: True | /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/train_labels.csv
sample_submission.csv 존재: True | /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/sample_submission.csv
필요한 RS18A 데이터 파일이 모두 있습니다.


## Step 3-1. baseline transform 준비하기


In [5]:

import torch
from torchvision import transforms as T

# baseline 버전: 기본 resize + tensor + normalize만 사용합니다.
# 데이터셋 정제도, RandomPerspective/Affine shear도 적용하지 않습니다.
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

valid_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = valid_transform
print("baseline transform 준비 완료")


baseline transform 준비 완료


Step 4. 데이터 전처리 개념 이해하기


In [6]:

from pathlib import Path
import pandas as pd

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
TRAIN_LABELS_PATH = DATA_DIR / "train_labels.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

print("DATA_DIR:", DATA_DIR)
print("TRAIN_DIR:", TRAIN_DIR.exists())
print("TEST_DIR:", TEST_DIR.exists())
print("TRAIN_LABELS_PATH:", TRAIN_LABELS_PATH.exists())
print("SAMPLE_SUBMISSION_PATH:", SAMPLE_SUBMISSION_PATH.exists())

print("train 이미지 개수:", len(list(TRAIN_DIR.glob("*.jpg"))))
print("test 이미지 개수:", len(list(TEST_DIR.glob("*.jpg"))))

train_df = pd.read_csv(TRAIN_LABELS_PATH)
sample_a = pd.read_csv(SAMPLE_SUBMISSION_PATH)

display(train_df.head())
display(sample_a.head())


DATA_DIR: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A
TRAIN_DIR: True
TEST_DIR: True
TRAIN_LABELS_PATH: True
SAMPLE_SUBMISSION_PATH: True
train 이미지 개수: 1366
test 이미지 개수: 468


,id,label
0,train_00000,0
1,train_00001,1
2,train_00002,0
3,train_00003,1
4,train_00004,1


,id,dusty_prob
0,test_00000,0.5
1,test_00001,0.5
2,test_00002,0.5
3,test_00003,0.5
4,test_00004,0.5


Step 5. 실행 스크립트가 있는지 확인


In [7]:

print("외부 실행 스크립트 확인 단계는 제거했습니다.")
print("이 노트북은 아래 데이터만 있으면 실행됩니다.")
print("-", DATA_DIR / "train")
print("-", DATA_DIR / "test")
print("-", DATA_DIR / "train_labels.csv")
print("-", DATA_DIR / "sample_submission.csv")


외부 실행 스크립트 확인 단계는 제거했습니다.
이 노트북은 아래 데이터만 있으면 실행됩니다.
- /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/train
- /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/test
- /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/train_labels.csv
- /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/sample_submission.csv


원본 데이터만 있으면 학습과 예측이 가능하다. 별도 보조 스크립트나 정제 CSV는 사용하지 않는다.


Step 6. baseline 학습 CSV 만들기


In [8]:

import pandas as pd

# baseline 평가는 원본 train_labels.csv 전체를 기준으로 합니다.
baseline_train = train_df.copy()
baseline_train.to_csv(BASELINE_TRAIN_CSV_PATH, index=False)

print("baseline_train shape:", baseline_train.shape)
print("columns:", baseline_train.columns.tolist())
print("저장 경로:", BASELINE_TRAIN_CSV_PATH)

display(baseline_train.head())

print("\nlabel 분포:")
print(baseline_train["label"].value_counts().sort_index())


baseline_train shape: (1366, 2)
columns: ['id', 'label']
저장 경로: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_individual/baseline_train_labels.csv


,id,label
0,train_00000,0
1,train_00001,1
2,train_00002,0
3,train_00003,1
4,train_00004,1



label 분포:
0    813
1    553
Name: label, dtype: int64


이 과정에서 봐야 하는 것


Step 7. 원본 train과 baseline 학습 CSV 비교하기: 원본 1366장을 모두 쓰는지 확인한다.


In [9]:

print("원본 train 개수:", len(train_df))
print("baseline 학습 CSV 개수:", len(baseline_train))
print("제외된 개수:", len(train_df) - len(baseline_train))

print("\n학습 대상 label 분포:")
print(baseline_train["label"].value_counts().sort_index())


원본 train 개수: 1366
baseline 학습 CSV 개수: 1366
제외된 개수: 0

학습 대상 label 분포:
0    813
1    553
Name: label, dtype: int64


원본 train_labels.csv를 그대로 사용한다. sample_weight나 exclude_reason 같은 정제용 컬럼은 추가하지 않는다.


Step 8. 학습 경로 변수 만들기:
노트북 내부 학습/예측 코드가 읽을 데이터 경로와 출력 경로를 준비


In [10]:

train_dir = str(TRAIN_DIR)
test_dir = str(TEST_DIR)
sample_submission_path = str(SAMPLE_SUBMISSION_PATH)
train_csv_path = str(BASELINE_TRAIN_CSV_PATH)
model_path = str(MODEL_PATH)
submission_path = str(SUBMISSION_PATH)
python_executable = str(PYTHON_EXECUTABLE)

print("train_dir:", train_dir)
print("test_dir:", test_dir)
print("sample_submission_path:", sample_submission_path)
print("train_csv_path:", train_csv_path)
print("model_path:", model_path)
print("submission_path:", submission_path)
print("python_executable:", python_executable)


train_dir: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/train
test_dir: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/test
sample_submission_path: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/data/RS18A/sample_submission.csv
train_csv_path: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_individual/baseline_train_labels.csv
model_path: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_individual/model_individual.pt
submission_path: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_individual/submission_a_individual.csv
python_executable: /home/thkim0/venv/v1/bin/python


Step 9. Track A EfficientNet baseline 학습하기


In [11]:

import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models
from PIL import Image

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

class TrackADataset(Dataset):
    def __init__(self, df, image_dir, transform=None, has_label=True):
        self.df = df.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self.image_dir / f"{row['id']}.jpg"
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        if self.has_label:
            label = int(row["label"])
            return image, torch.tensor(label, dtype=torch.long)
        return image, row["id"]

def make_stratified_split(df, val_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    train_idx = []
    val_idx = []
    for label, group in df.groupby("label"):
        idx = group.index.to_numpy()
        rng.shuffle(idx)
        n_val = max(1, int(round(len(idx) * val_ratio)))
        val_idx.extend(idx[:n_val].tolist())
        train_idx.extend(idx[n_val:].tolist())
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx

train_idx, val_idx = make_stratified_split(baseline_train, VAL_RATIO, SEED)
full_train_ds = TrackADataset(baseline_train, TRAIN_DIR, transform=train_transform, has_label=True)
full_valid_ds = TrackADataset(baseline_train, TRAIN_DIR, transform=valid_transform, has_label=True)
train_ds = Subset(full_train_ds, train_idx)
valid_ds = Subset(full_valid_ds, val_idx)

num_workers = 2
pin_memory = device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

if USE_PRETRAINED:
    try:
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        print("EfficientNet-B0 pretrained weights 사용")
    except Exception as e:
        print("pretrained weights를 불러오지 못해 weights=None으로 진행:", repr(e))
        model = models.efficientnet_b0(weights=None)
else:
    model = models.efficientnet_b0(weights=None)
    print("EfficientNet-B0 weights=None 사용")

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item() * labels.size(0)
            val_correct += (logits.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)

    train_loss /= max(train_total, 1)
    val_loss /= max(val_total, 1)
    train_acc = train_correct / max(train_total, 1)
    val_acc = val_correct / max(val_total, 1)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    }
    history.append(row)
    print(
        f"epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss {train_loss:.4f} acc {train_acc:.4f} | "
        f"val_loss {val_loss:.4f} acc {val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "config": {
                "arch": "efficientnet_b0",
                "img_size": IMG_SIZE,
                "use_pretrained": USE_PRETRAINED,
                "batch_size": BATCH_SIZE,
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
            },
        }, MODEL_PATH)
        print("best model saved:", MODEL_PATH)

history_df = pd.DataFrame(history)
display(history_df)


device: cuda
gpu: NVIDIA GeForce RTX 4070 SUPER
EfficientNet-B0 weights=None 사용


/home/thkim0/venv/v1/lib/python3.10/site-packages/torch/nn/modules/conv.py:459: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,


epoch 01/5 | train_loss 0.6881 acc 0.5723 | val_loss 0.6864 acc 0.5949
best model saved: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_individual/model_individual.pt
epoch 02/5 | train_loss 0.6510 acc 0.6154 | val_loss 0.6348 acc 0.6642
best model saved: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_individual/model_individual.pt
epoch 03/5 | train_loss 0.6307 acc 0.6603 | val_loss 0.6982 acc 0.6314
epoch 04/5 | train_loss 0.5669 acc 0.7115 | val_loss 0.7020 acc 0.6642
epoch 05/5 | train_loss 0.4645 acc 0.7921 | val_loss 0.8338 acc 0.6679


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,0.688118,0.572344,0.686397,0.594891
1,2,0.650956,0.615385,0.634805,0.664234
2,3,0.630685,0.660256,0.698225,0.631387
3,4,0.566927,0.711538,0.701959,0.664234
4,5,0.464453,0.792125,0.833765,0.667883


원본 train_labels.csv 기반 baseline 학습이다. 데이터 정제나 추가 geometric augmentation 없이 모델 구조와 학습 흐름을 확인한다.


Step 10. 학습 결과 확인하기:
학습된 모델 파일이 실제로 생겼는지 확인한다.
학습이 끝났다고 해도 모델 파일이 저장되지 않았으면 예측을 못한다.


In [12]:

from pathlib import Path

model_path_obj = Path(model_path)

print("모델 저장 여부:", model_path_obj.exists())
print("모델 파일 크기:", model_path_obj.stat().st_size if model_path_obj.exists() else "없음")


모델 저장 여부: True
모델 파일 크기: 16341547


Step 11. test 이미지 예측하기 : 학습된 모델로 test 이미지의 Dusty 확률을 만든다.


In [13]:

import torch
from torch.utils.data import DataLoader

checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

sample_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)
test_ds = TrackADataset(sample_df, TEST_DIR, transform=test_transform, has_label=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=(device.type == "cuda"))

ids = []
probs = []

with torch.no_grad():
    for images, batch_ids in test_loader:
        images = images.to(device, non_blocking=True)
        logits = model(images)
        dusty_prob = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
        ids.extend(list(batch_ids))
        probs.extend(dusty_prob.tolist())

submission = pd.DataFrame({"id": ids, "dusty_prob": probs})
submission = sample_df[["id"]].merge(submission, on="id", how="left")
submission.to_csv(SUBMISSION_PATH, index=False)

print("저장 완료:", SUBMISSION_PATH)
display(submission.head())
print("shape:", submission.shape)
print("dusty_prob range:", submission["dusty_prob"].min(), submission["dusty_prob"].max())


저장 완료: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_individual/submission_a_individual.csv


,id,dusty_prob
0,test_00000,0.347810
1,test_00001,0.302664
2,test_00002,0.495523
3,test_00003,0.383780
4,test_00004,0.335988


shape: (468, 2)
dusty_prob range: 0.2167949229478836 0.958055317401886


test 이미지 468장에 대해 dusty_prob를 만든다. 이 결과는 원본 학습 데이터 기준 baseline 제출 파일이다.


Step 12. 제출 CSV 검증하기 : Kaggle에 올릴 수 있는 CSV인지 마지막으로 검사


In [14]:

submission = pd.read_csv(SUBMISSION_PATH)

display(submission.head())
print("shape:", submission.shape)
print("columns:", submission.columns.tolist())

print("\n결측치:")
print(submission.isnull().sum())

print("\ndusty_prob 범위:")
print(submission["dusty_prob"].min(), submission["dusty_prob"].max())
print("\n저장 경로:", SUBMISSION_PATH)


,id,dusty_prob
0,test_00000,0.347810
1,test_00001,0.302664
2,test_00002,0.495523
3,test_00003,0.383780
4,test_00004,0.335988


shape: (468, 2)
columns: ['id', 'dusty_prob']

결측치:
id            0
dusty_prob    0
dtype: int64

dusty_prob 범위:
0.2167949229478836 0.958055317401886

저장 경로: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_individual/submission_a_individual.csv


Step 13. CSV 경로 확인


In [15]:

print("로컬 환경에서는 별도 다운로드가 필요 없습니다.")
print("제출 CSV 경로:", SUBMISSION_PATH)


로컬 환경에서는 별도 다운로드가 필요 없습니다.
제출 CSV 경로: /home/thkim0/github/AIFFEL_quest_rs/MainQuest/Quest02/outputs/trackA_individual/submission_a_individual.csv
